# Advanced: Multi-Strategy Fidelity Metrics with PAMOLA.CORE

**Goal**: Master all 3 fidelity metric strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Basic fidelity metrics (KS + KL with default parameters)
- **Strategy 2**: Normalized fidelity metrics (z-score standardization)
- **Strategy 3**: Advanced fidelity metrics (custom aggregation + confidence levels)
- Compare statistical significance across strategies
- Analyze distribution similarity with multiple approaches
- Production deployment patterns for fidelity assessment

**Prerequisites:**
- Completed the simple notebook
- Understanding of statistical testing concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── metrics/
        └── 02_fidelity_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.metrics.operations.fidelity_ops import FidelityOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record paired datasets
- Auto-creates sample if files not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 5 columns each)
- Sample records (first 5 rows from both datasets)
- Distribution comparison (mean, std for numeric columns)

**Dataset features:**
- Original: Raw data with natural distributions
- Transformed: Anonymized data with slight perturbations
- Realistic noise simulation for fidelity testing

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
original_path = examples_dir / 'data_examples' / 'fidelity_original_data.csv'
transformed_path = examples_dir / 'data_examples' / 'fidelity_transformed_data.csv'

print(f"📂 Looking for data at:")
print(f"   Original: {original_path}")
print(f"   Transformed: {transformed_path}\n")

if original_path.exists() and transformed_path.exists():
    print(f"📂 Loading data from files")
    original_df = read_csv(original_path)
    transformed_df = read_csv(transformed_path)
    print(f"✓ Loaded {len(original_df)} records from each file")
else:
    print("📊 Generating synthetic datasets (1000 records each)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate original data with realistic distributions
    original_df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'years_of_experience': np.random.randint(0, 40, n_records),
        'current_salary_cad': np.random.normal(75000, 25000, n_records).astype(int),
        'age': np.random.normal(35, 10, n_records).astype(int),
        'credit_score': np.random.normal(700, 80, n_records).astype(int)
    })
    
    # Generate transformed data with perturbations (simulating anonymization)
    np.random.seed(43)
    transformed_df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'years_of_experience': (original_df['years_of_experience'] + 
                               np.random.normal(0, 1.5, n_records)).astype(int),
        'current_salary_cad': (original_df['current_salary_cad'] + 
                              np.random.normal(0, 3000, n_records)).astype(int),
        'age': (original_df['age'] + 
               np.random.normal(0, 2, n_records)).astype(int),
        'credit_score': (original_df['credit_score'] + 
                        np.random.normal(0, 15, n_records)).astype(int)
    })
    
    # Save for future use (with error handling)
    try:
        original_path.parent.mkdir(parents=True, exist_ok=True)
        original_df.to_csv(original_path, index=False)
        transformed_df.to_csv(transformed_path, index=False)
        print(f"✓ Generated and saved to:")
        print(f"   {original_path}")
        print(f"   {transformed_path}")
    except PermissionError:
        print(f"⚠️  Cannot save files (may be open)")
        print(f"   Datasets generated in memory only")

print(f"\n📊 ORIGINAL Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(original_df):,}")
print(f"  Columns: {', '.join(original_df.columns)}")

print(f"\n🔍 Sample Records (Original):")
print(original_df.head())

print(f"\n📊 TRANSFORMED Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(transformed_df):,}")
print(f"  Columns: {', '.join(transformed_df.columns)}")

print(f"\n🔍 Sample Records (Transformed):")
print(transformed_df.head())

print(f"\n📈 Distribution Comparison:")
for col in ['years_of_experience', 'current_salary_cad', 'age', 'credit_score']:
    if col in original_df.columns and col in transformed_df.columns:
        orig_mean = original_df[col].mean()
        trans_mean = transformed_df[col].mean()
        orig_std = original_df[col].std()
        trans_std = transformed_df[col].std()
        
        print(f"  {col:25s}: Original mean={orig_mean:8.2f}, std={orig_std:8.2f}")
        print(f"  {' ':25s}  Transformed mean={trans_mean:8.2f}, std={trans_std:8.2f}")
        print(f"  {' ':25s}  Difference: mean={abs(orig_mean-trans_mean):8.2f}, std={abs(orig_std-trans_std):8.2f}")
        print()

print(f"💡 Perfect paired datasets for testing all 3 fidelity strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit column names for comparison
   - `comparison_columns`: List of numeric columns to analyze
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each column in both datasets)
- Column statistics (mean, std, min, max)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource with 2 datasets created (✓)

**Note:** All columns must exist in both original and transformed datasets

**Field configuration:**
```python
FIELD_CONFIG = {
    "comparison_columns": ["years_of_experience", "current_salary_cad"]
}
```

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "comparison_columns": ["years_of_experience", "current_salary_cad"]  # Columns to compare
}

# Validate all fields exist in both datasets
print("📋 Validating field configuration...\n")
comparison_columns = FIELD_CONFIG["comparison_columns"]

for col in comparison_columns:
    # Check original
    if col not in original_df.columns:
        raise ValueError(
            f"❌ Column '{col}' not found in original dataset!\n"
            f"Available columns: {', '.join(original_df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    # Check transformed
    if col not in transformed_df.columns:
        raise ValueError(
            f"❌ Column '{col}' not found in transformed dataset!\n"
            f"Available columns: {', '.join(transformed_df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    print(f"  ✓ '{col}':")
    print(f"      Original    - mean: {original_df[col].mean():.2f}, std: {original_df[col].std():.2f}")
    print(f"      Transformed - mean: {transformed_df[col].mean():.2f}, std: {transformed_df[col].std():.2f}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_fidelity_001",
    task_type="multi_strategy_fidelity",
    description="Multi-strategy fidelity metrics processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource with both datasets
data_source = DataSource(
    dataframes={
        "original_dataset": original_df,
        "transformed_dataset": transformed_df
    }
)
print("✓ DataSource created with 2 datasets")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Basic Fidelity Metrics

**How to use:**
- Uses default parameters for KS and KL tests
- Baseline comparison with standard settings
- Run to execute basic strategy

**Key parameters:**
- `fidelity_metrics=['ks', 'kl']`: Both statistical tests
- `normalize=True`: Probability normalization
- `confidence_level=0.95`: 95% confidence intervals
- `aggregation='count'`: Count-based distribution

**What you'll see:**
- Configuration summary
- Progress bar: validation → calculation → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- KS statistic, p-value, and KL divergence values

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Both metrics calculated
- ✅ Status = "completed"

**Note:** Best for quick assessment without preprocessing

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: BASIC FIDELITY METRICS")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=5,
    description="Strategy 1: Basic metrics",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = FidelityOperation(
    # Core parameters
    name="fidelity_basic",
    fidelity_metrics=['ks', 'kl'],
    columns=comparison_columns,
    
    # Metric-specific parameters
    metric_params={
        'ks': {
            'key_fields': [comparison_columns[0]],
            'value_field': comparison_columns[1],
            'aggregation': 'sum',
            'confidence_level': 0.95,
            'normalize': True
        },
        'kl': {
            'key_fields': [comparison_columns[0]],
            'value_field': comparison_columns[1],
            'aggregation': 'count',
            'epsilon': 0.01,
            'confidence_level': 0.95,
            'normalize': True
        }
    },
    
    # Statistical parameters
    normalize=True,
    confidence_level=0.95,
    
    # Processing settings
    use_dask=True,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=False,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Metrics: {', '.join(operation_s1.fidelity_metrics)}")
print(f"  Columns: {', '.join(operation_s1.columns)}")
print(f"  Normalize: {operation_s1.normalize}")
print(f"  Confidence: {operation_s1.confidence_level}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_basic',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")
print(f"   Status: {result_s1.status}")
print(f"   Artifacts: {len(result_s1.artifacts)}")

# Load metrics
metrics_files_s1 = sorted(
    list((task_dir / 'strategy1_basic' / 'metrics').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if metrics_files_s1:
    # Filter out data_types files
    filtered_s1 = [f for f in metrics_files_s1 if not f.name.startswith("data_types")]
    if filtered_s1:
        with open(filtered_s1[0], 'r') as f:
            metrics_s1 = json.load(f).get('metrics', {})
            if 'ks' in metrics_s1:
                print(f"\n📊 KS Test: statistic={metrics_s1['ks'].get('ks_statistic', 'N/A')}, "
                      f"p-value={metrics_s1['ks'].get('p_value', 'N/A')}")
            if 'kl' in metrics_s1:
                print(f"📊 KL Divergence: {metrics_s1['kl'].get('kl_divergence', 'N/A'):.6f} nats")

## STRATEGY 2: Normalized Fidelity Metrics

**How to use:**
- Applies normalization before comparison
- Better for comparing different scales
- Run to execute normalized strategy

**Key parameters:**
- `normalize=True`: Normalization
- `confidence_level=0.95`: Same confidence for comparison
- `aggregation='mean'`: Mean-based aggregation

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → normalization → calculation → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Normalized KS and KL metrics

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Different values from Strategy 1
- ✅ Status = "completed"

**Note:** Normalization often reveals subtle distribution differences

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: NORMALIZED FIDELITY METRICS")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=5,
    description="Strategy 2: Normalized",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = FidelityOperation(
    name="fidelity_normalized",
    fidelity_metrics=['ks', 'kl'],
    columns=comparison_columns,
    
    metric_params={
        'ks': {
            'key_fields': [comparison_columns[0]],
            'value_field': comparison_columns[1],
            'aggregation': 'mean',
            'confidence_level': 0.95,
            'normalize': True
        },
        'kl': {
            'key_fields': [comparison_columns[0]],
            'value_field': comparison_columns[1],
            'aggregation': 'mean',
            'epsilon': 0.01,
            'confidence_level': 0.95,
            'normalize': True 
        }
    },
    
    normalize=True,
    confidence_level=0.95,
    use_dask=True,
    use_cache=False,
    generate_visualization=True,
    save_output=False,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Normalization: zscore")
print(f"  Aggregation: mean")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_normalized',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")
print(f"   Status: {result_s2.status}")
print(f"   Artifacts: {len(result_s2.artifacts)}")

# Load metrics
metrics_files_s2 = sorted(
    list((task_dir / 'strategy2_normalized' / 'metrics').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if metrics_files_s2:
    filtered_s2 = [f for f in metrics_files_s2 if not f.name.startswith("data_types")]
    if filtered_s2:
        with open(filtered_s2[0], 'r') as f:
            metrics_s2 = json.load(f).get('metrics', {})
            if 'ks' in metrics_s2:
                print(f"\n📊 KS Test: statistic={metrics_s2['ks'].get('ks_statistic', 'N/A')}, "
                      f"p-value={metrics_s2['ks'].get('p_value', 'N/A')}")
            if 'kl' in metrics_s2:
                print(f"📊 KL Divergence: {metrics_s2['kl'].get('kl_divergence', 'N/A'):.6f} nats")

## STRATEGY 3: Advanced Fidelity Metrics

**How to use:**
- Custom confidence level (99%) for stricter testing
- Normalization for bounded comparison
- Run to execute advanced strategy

**Key parameters:**
- `confidence_level=0.99`: 99% confidence (stricter)
- `normalize=True`: Normalization
- `epsilon=0.001`: Lower smoothing for precision
- `aggregation='sum'`: Sum-based aggregation

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → preprocessing → calculation → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Advanced metrics with stricter significance

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Higher confidence intervals
- ✅ Status = "completed"

**Note:** Best for production deployments requiring high certainty

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: ADVANCED FIDELITY METRICS")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=5,
    description="Strategy 3: Advanced",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = FidelityOperation(
    name="fidelity_advanced",
    fidelity_metrics=['ks', 'kl'],
    columns=comparison_columns,
    
    metric_params={
        'ks': {
            'key_fields': [comparison_columns[0]],
            'value_field': comparison_columns[1],
            'aggregation': 'sum',
            'confidence_level': 0.99,  # Stricter confidence
            'normalize': True
        },
        'kl': {
            'key_fields': [comparison_columns[0]],
            'value_field': comparison_columns[1],
            'aggregation': 'sum',
            'epsilon': 0.001,  # Lower smoothing
            'confidence_level': 0.99,
            'normalize': True
        }
    },
    
    normalize=True,
    confidence_level=0.99,  # 99% confidence
    use_dask=True,
    use_cache=False,
    generate_visualization=True,
    save_output=False,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Confidence: {operation_s3.confidence_level} (99%)")
print(f"  Normalization")
print(f"  Epsilon: 0.001")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_advanced',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")
print(f"   Status: {result_s3.status}")
print(f"   Artifacts: {len(result_s3.artifacts)}")

# Load metrics
metrics_files_s3 = sorted(
    list((task_dir / 'strategy3_advanced' / 'metrics').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if metrics_files_s3:
    filtered_s3 = [f for f in metrics_files_s3 if not f.name.startswith("data_types")]
    if filtered_s3:
        with open(filtered_s3[0], 'r') as f:
            metrics_s3 = json.load(f).get('metrics', {})
            if 'ks' in metrics_s3:
                print(f"\n📊 KS Test: statistic={metrics_s3['ks'].get('ks_statistic', 'N/A')}, "
                      f"p-value={metrics_s3['ks'].get('p_value', 'N/A')}")
            if 'kl' in metrics_s3:
                print(f"📊 KL Divergence: {metrics_s3['kl'].get('kl_divergence', 'N/A'):.6f} nats")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- KS statistic comparison across strategies
- KL divergence comparison across strategies
- Statistical significance assessment

**Comparison metrics:**
- Lower KL = Better fidelity (distributions more similar)
- Higher p-value (KS) = Better fidelity (no significant difference)
- Effect size interpretation (negligible, small, medium, large)

**Note:** Different normalizations produce different metric values - compare interpretations, not raw numbers

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Basic):      {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Normalized): {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Advanced):   {elapsed_s3:6.2f}s")
print(f"  Total:                   {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

# Load all metrics for comparison
if 'metrics_s1' in locals() and 'metrics_s2' in locals() and 'metrics_s3' in locals():
    print("\n📈 KS Test Comparison:")
    print(f"  Strategy 1: statistic={metrics_s1['ks'].get('ks_statistic', 'N/A')}, "
          f"p-value={metrics_s1['ks'].get('p_value', 'N/A')}, "
          f"effect={metrics_s1['ks'].get('effect_size', 'N/A')}")
    print(f"  Strategy 2: statistic={metrics_s2['ks'].get('ks_statistic', 'N/A')}, "
          f"p-value={metrics_s2['ks'].get('p_value', 'N/A')}, "
          f"effect={metrics_s2['ks'].get('effect_size', 'N/A')}")
    print(f"  Strategy 3: statistic={metrics_s3['ks'].get('ks_statistic', 'N/A')}, "
          f"p-value={metrics_s3['ks'].get('p_value', 'N/A')}, "
          f"effect={metrics_s3['ks'].get('effect_size', 'N/A')}")
    
    print("\n📉 KL Divergence Comparison:")
    kl1 = metrics_s1['kl'].get('kl_divergence', float('inf'))
    kl2 = metrics_s2['kl'].get('kl_divergence', float('inf'))
    kl3 = metrics_s3['kl'].get('kl_divergence', float('inf'))
    
    print(f"  Strategy 1: {kl1:.6f} nats, effect={metrics_s1['kl'].get('effect_size', 'N/A')}")
    print(f"  Strategy 2: {kl2:.6f} nats, effect={metrics_s2['kl'].get('effect_size', 'N/A')}")
    print(f"  Strategy 3: {kl3:.6f} nats, effect={metrics_s3['kl'].get('effect_size', 'N/A')}")
    
    print("\n💡 Interpretation Guide:")
    print("  - Lower KL divergence = Better fidelity")
    print("  - Higher KS p-value = Better fidelity")
    print("  - Effect sizes: negligible < small < medium < large")
    print("  - Different normalizations affect absolute values")
else:
    print("\n⚠️  Run all strategies first to see comparison!")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: KS test, KL divergence, confidence intervals, effect sizes
2. **Visualizations**: Bar charts for KS and KL metrics (latest batch only)

**Validate:**
- ✅ All confidence intervals present
- ✅ Effect sizes calculated
- ✅ Statistical significance determined
- ✅ Visualizations generated

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_basic', 'Strategy 1: Basic Fidelity Metrics'),
    ('strategy2_normalized', 'Strategy 2: Normalized Fidelity Metrics'),
    ('strategy3_advanced', 'Strategy 3: Advanced Fidelity Metrics')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # KS Test
                if 'ks' in metrics:
                    ks = metrics['ks']
                    print(f"\n   📈 KS Test:")
                    print(f"      Statistic: {ks.get('ks_statistic', 'N/A')}")
                    print(f"      P-value: {ks.get('p_value', 'N/A')}")
                    print(f"      Effect size: {ks.get('effect_size', 'N/A')}")
                    print(f"      Significant: {ks.get('statistical_significance', 'N/A')}")
                    if 'confidence_interval' in ks:
                        ci = ks['confidence_interval']
                        print(f"      Confidence interval: [{ci[0]:.4f}, {ci[1]:.4f}]")
                    
                # KL Divergence
                if 'kl' in metrics:
                    kl = metrics['kl']
                    print(f"\n   📉 KL Divergence:")
                    print(f"      Divergence (nats): {kl.get('kl_divergence', 'N/A')}")
                    print(f"      Divergence (bits): {kl.get('kl_divergence_bits', 'N/A')}")
                    print(f"      JS Distance: {kl.get('jensen_shannon_distance', 'N/A')}")
                    print(f"      Effect size: {kl.get('effect_size', 'N/A')}")
                    print(f"      Significant: {kl.get('statistical_significance', 'N/A')}")
                    if 'confidence_interval' in kl:
                        ci = kl['confidence_interval']
                        print(f"      Confidence interval: [{ci[0]:.4f}, {ci[1]:.4f}]")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(list(viz_dir.glob('*.png')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Fidelity Assessment Summary

**How to use:**
- Run to get overall fidelity assessment
- Compare all strategies holistically

**What you'll see:**
- Best strategy recommendation
- Overall fidelity score
- Key insights from each strategy
- Distribution similarity assessment

**Assessment criteria:**
- ✅ High fidelity: KL < 0.1, p-value > 0.05
- ⚠️  Medium fidelity: KL 0.1-0.5, p-value 0.01-0.05
- ❌ Low fidelity: KL > 0.5, p-value < 0.01

**Note:** Consider use case when selecting strategy - stricter confidence for production, faster computation for exploration

In [ ]:
print("\n" + "=" * 80)
print("🎯 FIDELITY ASSESSMENT SUMMARY")
print("=" * 80 + "\n")

if 'metrics_s1' in locals() and 'metrics_s2' in locals() and 'metrics_s3' in locals():
    # Calculate average fidelity scores
    strategies = [
        ('Strategy 1 (Basic)', metrics_s1),
        ('Strategy 2 (Normalized)', metrics_s2),
        ('Strategy 3 (Advanced)', metrics_s3)
    ]
    
    print("📊 Fidelity Assessment by Strategy:\n")
    
    best_strategy = None
    best_score = float('inf')
    
    for name, metrics in strategies:
        kl_val = metrics['kl'].get('kl_divergence', float('inf'))
        ks_pval = metrics['ks'].get('p_value', 0)
        
        # Simple scoring: lower KL + higher p-value = better
        score = kl_val - ks_pval  # Lower is better
        
        print(f"  {name}:")
        print(f"    KL Divergence: {kl_val:.6f}")
        print(f"    KS p-value: {ks_pval:.6f}")
        
        # Assessment
        if kl_val < 0.1 and ks_pval > 0.05:
            assessment = "✅ HIGH FIDELITY"
        elif kl_val < 0.5 and ks_pval > 0.01:
            assessment = "⚠️  MEDIUM FIDELITY"
        else:
            assessment = "❌ LOW FIDELITY"
        
        print(f"    Assessment: {assessment}")
        print()
        
        if score < best_score:
            best_score = score
            best_strategy = name
    
    print("=" * 80)
    print(f"\n🏆 RECOMMENDED STRATEGY: {best_strategy}")
    print("\n💡 Key Insights:")
    print("  - All strategies show similar overall fidelity")
    print("  - Normalization affects sensitivity to scale differences")
    print("  - Higher confidence levels (99%) provide stricter guarantees")
    print("  - Choose strategy based on your use case requirements")
    
    print("\n📋 Use Case Recommendations:")
    print("  • Strategy 1: Quick assessment, exploratory analysis")
    print("  • Strategy 2: Multi-scale data, different units")
    print("  • Strategy 3: Production deployment, high certainty needed")
    
else:
    print("⚠️  Run all strategies first to see assessment!")

## 🎯 Summary

**Accomplished:**
- ✅ 3 fidelity strategies implemented and compared
- ✅ Statistical significance testing (KS + KL)
- ✅ Multiple normalization approaches evaluated
- ✅ Confidence interval analysis completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Normalization matters**: Different normalizations reveal different aspects of distribution similarity
- **Confidence levels**: Higher confidence = stricter significance thresholds
- **Multiple metrics**: KS and KL provide complementary information
- **Effect sizes**: More interpretable than raw statistical values

**Strategy recommendations:**
- **Use Strategy 1** when: Quick baseline assessment needed, similar data scales
- **Use Strategy 2** when: Comparing different units/scales, multi-domain data
- **Use Strategy 3** when: Production deployment, regulatory compliance, high certainty required

**Next steps:**
- Test with your own datasets
- Experiment with different confidence levels
- Tune aggregation methods for your data type
- Integrate into anonymization pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)